# The Trading League — one week to beat the market
### EP0418 · optional, the whole class · opens in Lab 2 · entries close Friday 30 October, 23:59 · scored on the week of 2 to 6 November · results at Lab 4

**Scoreboard:** https://stanleysp-eng.github.io/EP0418-labs/
**Player's guide:** *The Trading League: a player's guide*, under Topic 2 on Brightspace. A
plan for the week, the ideas worth starting from, the three traps, and what a good
write-up looks like. Read it before Day 2.

**You need:** `ep0418.py`, which the notebook fetches, and an internet connection
for the prices.

---
A one-week competition with one rule that matters: **you are scored on prices
nobody has seen.**

The market is **SPY**, the fund that tracks the 500 largest US companies and the
most traded security in the world. The notebook fetches its last five years of
daily prices into your own Colab. You have until Friday night to build a rule on
them, with any AI tool, method or library you like. Then the market does what it
does for a week, and on the Saturday every entry is run on SPY's real prices from
Monday 2 to Friday 6 November, prices that did not exist when you entered. The
rule that looks best on the past is usually not the one that wins the week.
Finding out why is the point.

**The rules**

1. Your entry is a function `strategy(prices)` that returns a position for every
   trading day: 1 (hold SPY), 0 (cash) or -1 (short).
2. A decision for a day may use that day's prices and earlier. Never later.
3. The scorer is locked. It delays your position by one day, charges 0.1% of the
   amount traded each time you change position, and reports risk-adjusted return
   (the Sharpe ratio) after costs. Read it in Task 3; do not edit it.
4. Buy-and-hold is an entrant. Beating it after costs on the scored week is the prize.
5. Your strategy must change position at least four times over the five years.
   Buy-and-hold under a new name is not an entry.
6. The week is ranked on return after costs; five days is too few for a Sharpe
   ratio to mean anything, so the five-year scores show both.
7. Two podiums: best week, and best write-up (Task 6).
8. Enter as often as you like before the deadline; the latest file counts. The
   public scoreboard shows every entry's score on the five years you can all see.
   The scored week appears on the Saturday.

Paper trading only. Prices come from Yahoo Finance through **yfinance**, a free,
unofficial library for personal, educational use. Nothing here is investment
advice, and a live broker earns nothing.

## Putting the assistant's code into a TODO cell

A TODO cell has a line ending in `...`. Replace **only that line** with the
assistant's code, and keep the name on the left (`s`, `clean`, `H`). Delete any
`import` or `read_csv` lines it adds; the data is already loaded. If it used a
different name for the result, add one line at the end to rename it, e.g.
`s = hourly_df`.

---
## How every lab in this module works

You will **not** write this code from scratch. Ask your AI assistant to write it.
Your job is the part it cannot do: deciding what the right answer looks like,
reading what it gave you, and proving it did what you asked.

At the foot of each task is a **verification contract** — a few `check.*` lines.
**You may not edit them.** The AI can write any implementation it likes; it cannot
satisfy these by accident.

If a check fails, read the message. It tells you what was found and what to try
next. Failing checks are the normal state of a lab in progress, not a sign you
are behind.

> **First: File → Save a copy in Drive, and work in your copy.** This page opened from GitHub, and changes here are not saved. Reopen your copy from Google Drive next time.

## Set up Gemini for this notebook (once, 30 seconds)

Open Gemini (the spark icon at the bottom of the notebook). Then:

1. In its chat box, click the **sliders icon** next to **+** and choose **Configure**. The Notebook settings window opens.
2. Click **Add new custom instruction**. Name it **EP0418**, paste the text below into Instructions, click **Add to notebook**, then **Apply**.
3. Click the sliders icon again and choose **EP0418**, so the tick moves from Default to EP0418. That is the mode you work in.

Colab keeps the instruction with this notebook. Do the same once in each new notebook; the text is always in this cell.

```
I am new to coding. Explain any code you write in plain words: what it does to my data.
Before writing code for a bigger task, tell me your plan in plain words.
Never delete or drop rows unless I ask. If a step would remove rows, say so first.
After changing the data, show the row count before and after, and the smallest and largest values.
Codes such as -999 mean a missing reading. Point them out; do not treat them as real numbers.
Do not change any line that starts with check. Those are the lab's locked checks.
If you are not sure, say so. Do not guess.
```

**Why these seven lines.** Each one stops a mistake that AI assistants make with data, and that you would not notice until a check failed:

| Line | The mistake it prevents |
|---|---|
| Explain in plain words | Code you cannot account for. You will be asked what each step does to your data. |
| Plan first | Ten lines of code for the wrong task. The plan is quicker to correct than the code. |
| Never drop rows unless asked | `dropna()` quietly removing a third of a sensor file. |
| Row count and range before and after | The only cheap way to see what a step really did. |
| -999 is missing, not a number | An average of humidity that comes out negative. |
| Never touch `check.` lines | A test rewritten so that it passes. The checks are the contract. |
| Say so if unsure | A confident guess that reads exactly like a fact. |

**See it working (1 minute).** With EP0418 selected, ask Gemini:

> Write code to remove the rows of df where humidity_pct is -999.

It should warn you that this removes rows before it writes anything, treat -999 as a missing reading rather than a value, and explain the code in plain words. Now switch the tick to **Default**, ask the same thing in a new chat, and compare. Switch back to **EP0418**.

**Make it yours.** Click the sliders icon, choose Configure, and open EP0418 with the pencil icon. Add one line of your own, for example *Keep answers under ten lines* or *Always name the column you are using*, and Apply. Ask the same question again and see your line take effect. The seven course lines stay; yours go underneath.

In [ ]:
import os, urllib.request
REPO = "https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/"


def fetch(*names):
    """Download any course file that is not beside the notebook yet."""
    for name in names:
        if os.path.exists(name):
            continue
        try:
            urllib.request.urlretrieve(REPO + name, name)
            print("fetched", name)
        except Exception:
            print(f"Could not download {name}. Upload it with the folder icon on the left.")
            from google.colab import files
            files.upload()


fetch("ep0418.py")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from ep0418 import check, show


def filled(*values):
    """True once every TODO slot above has been replaced. Otherwise says so."""
    if any(v is ... or v is None for v in values):
        print("A TODO above is still '...'. Fill it in, then run this cell again.")
        return False
    return True

from ep0418 import backtest

if not os.path.exists("spy_daily.csv"):                  # fetched once, then reused
    try:
        import yfinance as yf          # if this fails: run  !pip install yfinance  first
        h = yf.Ticker("SPY").history(period="5y", interval="1d")
        pd.DataFrame({"Date": pd.to_datetime(h.index).tz_localize(None).normalize(),
                      "Close": h["Close"].values}).to_csv("spy_daily.csv", index=False)
    except Exception as e:
        print("Yahoo did not answer:", e, "- wait a minute and run this cell again.")

prices = pd.read_csv("spy_daily.csv", parse_dates=["Date"])
print(f"SPY: {len(prices):,} trading days, {prices.Date.min().date()} to {prices.Date.max().date()}, "
      f"latest ${prices.Close.iloc[-1]:,.2f}")
show.series(prices, col="Close", title="SPY closing price, five years")

position = strategy = None     # placeholders, so later cells give a readable message

## Task 1 — Put buy-and-hold on the board

Before anything clever: what does doing nothing earn?

In [ ]:
bh = backtest.buy_and_hold(prices)
print("Buy-and-hold, five years of SPY")
print(bh)

## Task 2 — A worked example, and what costs do to it

A moving-average crossover: hold SPY when the 20-day average is above the 100-day
average, otherwise cash. Run it twice, with and without costs. Then run it with a
5-day and 20-day pair, which trades far more often, and watch what happens to the
gap between the two scores.

In [ ]:
def crossover(prices, fast=20, slow=100):
    close = prices["Close"]
    f = close.rolling(fast).mean()
    s = close.rolling(slow).mean()
    return (f > s).astype(int).where(s.notna(), 0)     # 0 until the slow window fills

for fast, slow in [(20, 100), (5, 20)]:
    pos = crossover(prices, fast, slow)
    free = backtest.run(prices, pos, cost=0)
    paid = backtest.run(prices, pos)
    print(f"crossover {fast}/{slow}: {paid.trades} trades  "
          f"Sharpe {free.sharpe:.2f} without costs, {paid.sharpe:.2f} with costs  "
          f"(total {free.total:+.1%} -> {paid.total:+.1%})")

**Write down:** which pair trades more, and how much of its return the costs ate.
A rule that trades every few days has to be a lot better than one that trades
every few months, just to pay for itself.

## Task 3 — Read the scorer before you trust it

You are about to spend weeks trying to beat this function. Read it. The cell
prints its source. Then answer, in the cell below, in your own words:

1. Which day's price move does the position you choose on Monday earn?
2. You are long (1) and switch to short (-1). How many units did you trade, and
   what did that cost?
3. Why does the scorer shift your position rather than trusting you to?

In [ ]:
import inspect
print(inspect.getsource(backtest.run))

**My answers:**

1.
2.
3.

## Task 4 — Your strategy

Replace the body of `strategy`. It gets the price table and must return one
position per row. Ask your assistant; then read what it wrote, and run the
checks. `check.no_lookahead` removes the end of the file and reruns your
function. If any earlier decision changes, you were using the future.

> I have a pandas DataFrame `prices` with columns Date and Close, one row per
> trading day. Write a Python function `strategy(prices)` that returns a pandas
> Series of the same length holding 1, 0 or -1 for each day: a trading position
> decided using only that day's row and earlier rows. Use [describe your idea
> here]. Fill the rows before any rolling window is full with 0. Do not use any
> information from later rows. Explain in plain words what it does.

Ideas that are honest to start from: a longer or shorter crossover; a breakout
above the highest close of the last n days; a mean-reversion rule that buys after
a fall of more than x%; a rule that goes to cash when recent volatility is high.

In [ ]:
def strategy(prices):
    # TODO(you): replace this body with your rule. Return 1, 0 or -1 for every row.
    position = ...
    return position


if strategy(prices.head(150)) is not ...:
    position = strategy(prices)

# ---- verification contract — do not edit ----
check.position_is_valid(position, prices)
check.no_lookahead(strategy, prices)
check.trades_at_least(position, minimum=4)
check.report()

In [ ]:
if filled(position):
    mine = backtest.run(prices, position)
    print("Your strategy, five years of SPY"); print(mine)
    print("\nBuy-and-hold"); print(bh)
    check.compare_to_baseline(mine.sharpe, bh.sharpe, metric="Sharpe", lower_is_better=False)
    check.report()
    show.flags(prices.set_index("Date").Close, pd.Series(position).values == 1,
               title="SPY, five years; marked where your rule is long")

### If you got a NOTE

Your strategy did not beat buy-and-hold on the past five years. Most do not; SPY
has spent most of that time going up. That costs you nothing here, and it does not
mean it will lose on the hidden weeks either. Write down what buy-and-hold is
capturing that your rule is not.

## Task 5 — Run your own hidden week first

You cannot see next week. You can imitate it. Pretend the file ends six months
ago: tune your rule on the rows before that date, then score it on the rows
after. The second number is the honest one. Then do it again ending one week ago,
and look at how much a single week can differ from the long run.

If you change your rule after seeing the second number, you have just made the
second number part of the data you tuned on. Do it once, write it down, and
change the rule only if you can say why it should work, not because it did.

In [ ]:
if filled(position):
    cut = prices.Date < prices.Date.max() - pd.Timedelta(days=182)
    before = backtest.run(prices[cut], strategy(prices[cut]))
    after_pos = strategy(prices).loc[~cut]           # decided with the full history, scored on the tail
    after = backtest.run(prices[~cut], after_pos)
    bh_after = backtest.buy_and_hold(prices[~cut])
    print(f"tuned on   {cut.sum():,} days: Sharpe {before.sharpe:.2f}  total {before.total:+.1%}")
    print(f"scored on  {(~cut).sum():,} days: Sharpe {after.sharpe:.2f}  total {after.total:+.1%}"
          f"   (buy-and-hold on the same days: Sharpe {bh_after.sharpe:.2f}, total {bh_after.total:+.1%})")
    week = prices.Date >= prices.Date.max() - pd.Timedelta(days=7)
    last_week = backtest.run(prices[week], strategy(prices).loc[week]); bh_week = backtest.buy_and_hold(prices[week])
    print(f"last week  {int(week.sum())} days: return {last_week.total:+.2%}   (buy-and-hold {bh_week.total:+.2%})")

**Did the second number hold up?** If the Sharpe ratio halved or changed sign,
your rule was fitted to the first period rather than to anything real. Say so
below; the write-up prize is for honesty, not for a high number.

## Task 6 — The write-up (half a page, the second podium)

Answer in the cell below. This is what the tutor reads, and it is the part the
module actually grades in CA3.

1. **What the rule assumes about the market.** One or two sentences.
2. **Where it will fail.** The kind of period in which it loses, and why.
3. **Your own hidden-period test.** The two numbers from Task 5, and what you
   concluded.
4. **One mistake the AI assistant made** while writing or changing this rule, how
   you caught it, and what you changed. Everyone has one.

**Write-up**

1.
2.
3.
4.

## Submit

Fill in `ADMIN_NO` and run the cell. It saves your function and write-up as one
file and downloads it. Upload that file to the League folder on Brightspace.
Resubmitting replaces your earlier entry. Entries close Friday 30 October at 23:59.

In [ ]:
ADMIN_NO = "..."          # TODO(you): your admin number, e.g. 2512345
WRITEUP = """
TODO(you): paste your four write-up answers here.
"""

if ADMIN_NO != "..." and filled(position):
    import inspect
    try:
        source = inspect.getsource(strategy)
    except OSError:
        source = None
        print("Could not read the function's source. Run the cell that defines "
              "strategy() once more, then run this cell again.")
    out = f"strategy_{ADMIN_NO}.py"
    with open(out, "w") as f:
        f.write('"""' + WRITEUP.strip() + '\n"""\n\n')
        f.write("import numpy as np\nimport pandas as pd\n\n\n")
        f.write(source or "# strategy() source missing: see the message above\n")
    print("saved", out)
    try:
        from google.colab import files
        files.download(out)
    except Exception:
        print("Download it from the folder icon on the left.")

---
## Before Lab 4

- The AI mistake from the write-up, below.
- Your Task 5 numbers written down before you saw the scoreboard.

At Lab 4 the tutor shows two tables: the scoreboard on the five years you could
see, and the scoreboard on the week of 2 to 6 November. Look at who moved, and
which way. That reshuffle is the whole lesson of the league.

**Mistake caught:** *(what I expected · what was there · how I found it)*